In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker
fake = Faker('en_IN') # Using Indian locale for realistic company names
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# --- Configuration ---
NUM_NORMAL_TXNS = 5000
NUM_ANOMALIES = 50
NUM_BENFORD_FRAUD = 200

print("Generating LedgerSpy Synthetic Audit Data...")

# 1. Create Base Entities
source_account = "Main Operating A/C - HDFC"
legit_vendors = [fake.company() + " " + fake.company_suffix() for _ in range(50)]

# Inject Ghost Vendors (Slightly altered names for the Fuzzy Matcher to catch)
ghost_vendors = [
    legit_vendors[0].replace("Ltd", "L.T.D"),
    legit_vendors[1] + " Pvt",
    legit_vendors[2].replace(" ", "")
]
all_vendors = legit_vendors + ghost_vendors

# --- Generate Normal Transactions ---
data = []
start_date = datetime(2025, 1, 1)

for _ in range(NUM_NORMAL_TXNS):
    vendor = random.choice(legit_vendors)
    # Normal amounts between ₹500 and ₹50,000
    amount = round(random.uniform(500, 50000), 2)
    # Normal business hours (9 AM to 6 PM)
    hour = random.randint(9, 18)
    day_offset = random.randint(0, 365)
    txn_date = start_date + timedelta(days=day_offset, hours=hour, minutes=random.randint(0, 59))
    
    # Skip weekends for normal business
    if txn_date.weekday() >= 5: 
        txn_date -= timedelta(days=2)

    data.append([f"TXN-{fake.unique.random_number(digits=6)}", txn_date, amount, source_account, vendor, "Normal"])

# --- Inject Fraud Type 1: The Outliers (For the Isolation Forest) ---
# Massive amounts, weird hours (e.g., 3 AM on a Sunday)
for _ in range(NUM_ANOMALIES):
    vendor = random.choice(legit_vendors)
    amount = round(random.uniform(500000, 2000000), 2) # Massive amounts
    txn_date = start_date + timedelta(days=random.randint(0, 365), hours=random.randint(1, 4)) # 1 AM to 4 AM
    
    # Force it to be a Sunday
    days_to_sunday = 6 - txn_date.weekday()
    txn_date += timedelta(days=days_to_sunday)

    data.append([f"TXN-{fake.unique.random_number(digits=6)}", txn_date, amount, source_account, vendor, "Anomaly"])

# --- Inject Fraud Type 2: Benford's Law Evasion ---
# Fraudsters making up numbers often overuse 7s, 8s, and 9s.
for _ in range(NUM_BENFORD_FRAUD):
    vendor = random.choice(legit_vendors)
    # Force amounts to start with 8 or 9 (e.g., 8453.20, 9912.50)
    leading_digit = random.choice([8, 9])
    amount = round(leading_digit * 1000 + random.uniform(10, 999), 2)
    txn_date = start_date + timedelta(days=random.randint(0, 365), hours=random.randint(9, 17))
    
    data.append([f"TXN-{fake.unique.random_number(digits=6)}", txn_date, amount, source_account, vendor, "Benford Violation"])

# --- Inject Fraud Type 3: The Ghost Vendors ---
for _ in range(30): # 30 transactions to our fake shell companies
    vendor = random.choice(ghost_vendors)
    amount = round(random.uniform(10000, 45000), 2)
    txn_date = start_date + timedelta(days=random.randint(0, 365), hours=random.randint(9, 17))
    
    data.append([f"TXN-{fake.unique.random_number(digits=6)}", txn_date, amount, source_account, vendor, "Ghost Vendor"])

# --- Compile and Save ---
df = pd.DataFrame(data, columns=['transaction_id', 'timestamp', 'amount', 'source_entity', 'destination_entity', 'fraud_label'])

# Shuffle the dataframe so fraud isn't all at the bottom
df = df.sample(frac=1).reset_index(drop=True)

# Save to our root directory so the models can access it
csv_path = "../synthetic_ledger_data.csv"
df.to_csv(csv_path, index=False)

print(f"✅ Success! Generated {len(df)} rows of synthetic ledger data.")
print(f"📊 Normal TXNs: {NUM_NORMAL_TXNS}")
print(f"🚨 Anomalies Injected: {NUM_ANOMALIES}")
print(f"🔢 Benford Violations Injected: {NUM_BENFORD_FRAUD}")
print(f"👻 Ghost Vendor TXNs Injected: 30")
print(f"\nData saved to: {csv_path}")
df.head()